# Data and model loader logic

In [ ]:
import joblib
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [ ]:
# --- Notebook parameters ---

MODELS_DIR = "models"
DATA_DIR = "data"
SEED = 42


# --- Helper functions ---
def gini(y_true, y_score):
    """Gini coefficient = 2 * AUC - 1"""
    return 2.0 * roc_auc_score(y_true, y_score) - 1.0

In [ ]:
# --- Load test data ---

test_df = pd.read_parquet(f"{DATA_DIR}/df_test_preprocessed_cutoff44.parquet")

id_cols = ["month_decision", "weekday_decision", "WEEK_NUM", "case_id"]

test_df = test_df.sort_values("WEEK_NUM", kind="mergesort").reset_index(drop=True)
week_num_test = test_df["WEEK_NUM"].copy()

test_df = test_df.drop(columns=id_cols)

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

print(f"Test shape: {X_test.shape}")
print(f"Weeks: {week_num_test.min()} - {week_num_test.max()} "
      f"({week_num_test.nunique()} unieke weken)")
print(f"Default rate: {y_test.mean():.4f} ({int(y_test.sum())} positieven)")


In [ ]:
# --- Load models ---

# Logistic Regression
lr_best = joblib.load(f"{MODELS_DIR}/lr_best_44.joblib")

# XGBoost
xgb_best = XGBClassifier()
xgb_best.load_model(f"{MODELS_DIR}/xgb_best_44.json")

# MLP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp_best = torch.jit.load(f"{MODELS_DIR}/mlp_best_44.pt", map_location=device).eval()

mlp_weights = [v for k, v in mlp_best.state_dict().items() if k.endswith("weight")]
mlp_n_features = mlp_weights[0].shape[1]
mlp_hidden = [w.shape[0] for w in mlp_weights[:-1]]

print("LR :", f"C={lr_best.C:.6g}, l1_ratio={lr_best.l1_ratio}, "
              f"n_features={lr_best.n_features_in_}")
print("XGB:", f"n_trees={xgb_best.get_booster().num_boosted_rounds()}, "
              f"n_features={xgb_best.n_features_in_}")
print("MLP:", f"hidden={mlp_hidden}, n_features={mlp_n_features}")
print("Device:", device)
